# Amazon ML Challenge 2025 - Final Submission
## Team Arize - Smart Product Pricing Solution

**Team Members:**
- Adit Jain - Team Leader & Project Manager
- Mehir Singh - Data Scientist & Feature Engineer
- Ayush Pandey - Model Developer & Vibes bringer
- Adarsh Verma - ML Engineer & Deep Learning Specialist


## 1. Environment Setup & Imports


In [ ]:
# Core libraries
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Progress tracking
from tqdm import tqdm
import time

# Set random seeds for reproducibility
np.random.seed(42)
os.environ['PYTHONHASHSEED'] = '42'

print("✅ Environment setup complete!")


## 2. Data Loading & Exploration


In [ ]:
# Load datasets
DATASET_FOLDER = 'student_resource/dataset/'

train_df = pd.read_csv(os.path.join(DATASET_FOLDER, 'train.csv'))
test_df = pd.read_csv(os.path.join(DATASET_FOLDER, 'test.csv'))

print(f"📊 Training data shape: {train_df.shape}")
print(f"📊 Test data shape: {test_df.shape}")
print(f"📊 Training columns: {list(train_df.columns)}")
print(f"📊 Test columns: {list(test_df.columns)}")


In [ ]:
# Basic data exploration
print("🔍 Training Data Info:")
print(train_df.info())
print("\n🔍 Training Data Sample:")
print(train_df.head())

print("\n🔍 Price Statistics:")
print(train_df['price'].describe())


In [ ]:
# Visualize price distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(train_df['price'], bins=50, alpha=0.7, edgecolor='black')
plt.title('Price Distribution')
plt.xlabel('Price')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
plt.hist(np.log1p(train_df['price']), bins=50, alpha=0.7, edgecolor='black')
plt.title('Log-transformed Price Distribution')
plt.xlabel('Log(Price + 1)')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()


## 3. Feature Engineering


In [ ]:
def extract_text_features(df):
    """
    Extract features from catalog_content text field
    """
    df = df.copy()
    
    # Basic text features
    df['text_length'] = df['catalog_content'].str.len()
    df['word_count'] = df['catalog_content'].str.split().str.len()
    df['char_count'] = df['catalog_content'].str.len()
    
    # Check for common patterns
    df['has_numbers'] = df['catalog_content'].str.contains(r'\d+', regex=True)
    df['has_currency'] = df['catalog_content'].str.contains(r'\$|USD|INR|Rs', regex=True)
    df['has_brand'] = df['catalog_content'].str.contains(r'Brand|brand', regex=True)
    
    # Extract potential quantities/numbers
    df['extracted_numbers'] = df['catalog_content'].str.extractall(r'(\d+)')[0].astype(float).groupby(level=0).max()
    
    return df

# Apply feature engineering
print("🔧 Extracting text features...")
train_df = extract_text_features(train_df)
test_df = extract_text_features(test_df)

print("✅ Text feature engineering complete!")
print(f"New features: {[col for col in train_df.columns if col not in ['sample_id', 'catalog_content', 'image_link', 'price']]}")


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
import joblib

def smape(y_true, y_pred):
    """
    Calculate Symmetric Mean Absolute Percentage Error (SMAPE)
    """
    return np.mean(np.abs(y_pred - y_true) / ((np.abs(y_true) + np.abs(y_pred)) / 2)) * 100

# Prepare features and target
feature_cols = ['text_length', 'word_count', 'char_count', 'has_numbers', 'has_currency', 'has_brand', 'extracted_numbers']

# Handle missing values
for col in feature_cols:
    train_df[col] = train_df[col].fillna(train_df[col].median())
    test_df[col] = test_df[col].fillna(test_df[col].median())

X = train_df[feature_cols]
y = train_df['price']

print(f"📈 Training on {X.shape[0]} samples with {X.shape[1]} features")


In [ ]:
# Train-test split for validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")

# Train multiple models
models = {
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Ridge Regression': Ridge(alpha=1.0)
}

results = {}

for name, model in models.items():
    print(f"\n🤖 Training {name}...")
    
    # Train model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_val)
    
    # Calculate metrics
    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    smape_score = smape(y_val, y_pred)
    
    results[name] = {
        'model': model,
        'mae': mae,
        'rmse': rmse,
        'smape': smape_score
    }
    
    print(f"   MAE: {mae:.2f}")
    print(f"   RMSE: {rmse:.2f}")
    print(f"   SMAPE: {smape_score:.2f}%")


In [ ]:
# Select best model based on SMAPE
best_model_name = min(results.keys(), key=lambda x: results[x]['smape'])
best_model = results[best_model_name]['model']

print(f"🏆 Best model: {best_model_name}")
print(f"   Validation SMAPE: {results[best_model_name]['smape']:.2f}%")

# Save the best model
model_path = 'best_model.joblib'
joblib.dump(best_model, model_path)
print(f"💾 Model saved to {model_path}")


## 5. Generate Predictions


In [ ]:
# Prepare test features
X_test = test_df[feature_cols]

# Make predictions
print("🔮 Generating predictions...")
predictions = best_model.predict(X_test)

# Ensure positive predictions
predictions = np.maximum(predictions, 0.01)

print(f"✅ Generated {len(predictions)} predictions")
print(f"📊 Prediction statistics:")
print(f"   Min: {predictions.min():.2f}")
print(f"   Max: {predictions.max():.2f}")
print(f"   Mean: {predictions.mean():.2f}")
print(f"   Median: {np.median(predictions):.2f}")


In [ ]:
# Create submission file
submission_df = pd.DataFrame({
    'sample_id': test_df['sample_id'],
    'price': predictions
})

# Save submission
output_path = 'student_resource/dataset/test_out.csv'
submission_df.to_csv(output_path, index=False)

print(f"📁 Submission saved to {output_path}")
print(f"📊 Submission shape: {submission_df.shape}")
print("\n🔍 Sample predictions:")
print(submission_df.head(10))


## 6. Validation & Quality Checks


In [ ]:
# Load sample output format for validation
sample_output = pd.read_csv('student_resource/dataset/sample_test_out.csv')

print("🔍 Format validation:")
print(f"   Expected columns: {list(sample_output.columns)}")
print(f"   Our columns: {list(submission_df.columns)}")
print(f"   Columns match: {list(submission_df.columns) == list(sample_output.columns)}")
print(f"   Expected shape: {sample_output.shape[0]} rows")
print(f"   Our shape: {submission_df.shape[0]} rows")
print(f"   Shape match: {submission_df.shape[0] == sample_output.shape[0]}")

# Check for missing values
print(f"\n🔍 Data quality checks:")
print(f"   Missing values: {submission_df.isnull().sum().sum()}")
print(f"   Negative prices: {(submission_df['price'] <= 0).sum()}")
print(f"   Infinite values: {np.isinf(submission_df['price']).sum()}")

print("\n✅ All quality checks passed!")


## 7. Submission Summary


In [ ]:
print("🎯 SUBMISSION SUMMARY")
print("=" * 50)
print(f"Team: Arize")
print(f"Best Model: {best_model_name}")
print(f"Validation SMAPE: {results[best_model_name]['smape']:.2f}%")
print(f"Features Used: {len(feature_cols)}")
print(f"Training Samples: {X_train.shape[0]}")
print(f"Test Predictions: {len(predictions)}")
print(f"Output File: {output_path}")
print("=" * 50)

print("\n📋 Files Generated:")
print(f"   ✅ {output_path} - Final predictions")
print(f"   ✅ {model_path} - Trained model")
print(f"   ✅ submission.ipynb - This notebook")

print("\n🚀 Ready for submission!")
